# AI012 – Role C: Anomaly Definition and Label Builder

**Author:** Role C (Preetham Chandu)  
**Dataset:** `anomaly_detection_hourly_2020_2024.csv` (built by Sunain – Role A)  
**Task:** AI012 – Anomaly Detection Model | Project Phoenix

---

## What this notebook does

This notebook demonstrates the full Role C pipeline:

```
load_data → engineer_features → build_labels → (optional) save_outputs
```

It produces **two DataFrames**:

| Output | Used by | Purpose |
|--------|---------|--------|
| `features_df` | Role D (Isolation Forest) | Model training input |
| `labels_df` | Role F (Evaluation) | Ground truth for measuring model performance |

> **Important:** Labels are for **evaluation only**. They are never passed into the unsupervised model (Isolation Forest). The model finds anomalies on its own — Role C labels measure how well it did.

## 1. Setup

In [3]:
import sys
from pathlib import Path
import pandas as pd

# Add src to path so we can import label_builder
sys.path.append(str(Path.cwd().parent / 'src'))

from labeling.label_builder import load_data, engineer_features, build_labels, save_outputs, run

## 2. Load the dataset (Role A – Sunain's merged FIRMS + URLhaus data)

In [4]:
# Resolve path to the dataset in the repo
repo_root = Path.cwd().parents[3]  # Phoenix/
dataset_path = repo_root / 'ai-ml' / 'datasets' / 'anomaly_detection_hourly_2020_2024.csv'

df = load_data(str(dataset_path))

[label_builder] Loading dataset: anomaly_detection_hourly_2020_2024.csv
[label_builder] Loaded 169,539 rows, 45 columns


In [5]:
# Preview key columns
df[['time_window','region_id','firms_event_count','firms_avg_frp',
    'firms_avg_confidence','urlhaus_event_count','urlhaus_online_count',
    'urlhaus_unique_url_count']].head(3)

,time_window,region_id,firms_event_count,firms_avg_frp,firms_avg_confidence,urlhaus_event_count,urlhaus_online_count,urlhaus_unique_url_count
0,2020-01-01 03:00:00,lat_-3_lon_28,98,6.496224,59.693878,0,0,0
1,2020-01-01 03:00:00,lat_-4_lon_27,5,10.098000,60.000000,0,0,0
2,2020-01-01 03:00:00,lat_-4_lon_28,368,8.107690,61.141304,0,0,0


## 3. Engineer features for anomaly detection (Role D input)

In [6]:
features_df = engineer_features(df)
features_df[['firms_event_count','firms_avg_frp','firms_avg_brightness',
             'firms_avg_confidence','urlhaus_event_count','urlhaus_unique_url_count',
             'urlhaus_online_count','cyber_to_hazard_ratio','total_cyber_signal']].head(3)

[label_builder] Feature set prepared: 13 features, 169,539 records


,firms_event_count,firms_avg_frp,firms_avg_brightness,firms_avg_confidence,urlhaus_event_count,urlhaus_unique_url_count,urlhaus_online_count,cyber_to_hazard_ratio,total_cyber_signal
0,98,6.496224,345.226939,59.693878,0,0,0,0.0,0
1,5,10.098000,345.386000,60.000000,0,0,0,0.0,0
2,368,8.107690,348.173668,61.141304,0,0,0,0.0,0


## 4. Build anomaly labels (Role F evaluation input)

In [7]:
labels_df = build_labels(df)

[label_builder] Rule results:
[label_builder]   extreme_fire_intensity                     1,696 rows flagged
[label_builder]   high_confidence_extreme_fire               3,034 rows flagged
[label_builder]   cyber_spike_low_hazard                     1,593 rows flagged
[label_builder]   hazard_cyber_mismatch                      8,418 rows flagged
[label_builder]   active_online_threats                      2,939 rows flagged
[label_builder]   rare_region_active                            68 rows flagged
[label_builder]   cyber_volume_spike                         2,939 rows flagged

[label_builder] Label summary:
  Total rows : 169,539
  Anomalies  : 11,419  (6.74%)
  Normal     : 158,120


In [8]:
# Sample anomalous rows
labels_df[labels_df['anomaly_flag'] == 1].head(5)

,time_window,region_id,anomaly_flag,anomaly_score,anomaly_reason
3,2020-01-01 03:00:00,lat_-4_lon_29,1,0.1429,hazard_cyber_mismatch
10,2020-01-01 03:00:00,lat_-7_lon_29,1,0.4286,extreme_fire_intensity | high_confidence_extre...
12,2020-01-01 03:00:00,lat_-8_lon_28,1,0.2857,high_confidence_extreme_fire | hazard_cyber_mi...
15,2020-01-01 03:00:00,lat_-9_lon_29,1,0.1429,hazard_cyber_mismatch
22,2020-01-01 04:00:00,lat_-4_lon_29,1,0.1429,hazard_cyber_mismatch


In [9]:
print('Anomaly flag breakdown:')
print(labels_df['anomaly_flag'].value_counts().to_string())
print()
print('Top anomaly reasons:')
print(labels_df['anomaly_reason'].value_counts().head(8).to_string())

Anomaly flag breakdown:
anomaly_flag
0    158120
1     11419

Top anomaly reasons:
anomaly_reason
none                                                                             158120
hazard_cyber_mismatch                                                              4600
high_confidence_extreme_fire | hazard_cyber_mismatch                               2125
cyber_spike_low_hazard | active_online_threats | cyber_volume_spike                1592
active_online_threats | cyber_volume_spike                                         1327
extreme_fire_intensity | high_confidence_extreme_fire | hazard_cyber_mismatch       890
extreme_fire_intensity | hazard_cyber_mismatch                                      798
rare_region_active                                                                   62


## 5. Anomaly rate — contamination guidance for Role D

The **6.74% anomaly rate** is intentional. It falls within the 1–10% contamination range that the AI012 task guide specifies for Isolation Forest tuning. Role D should test `contamination` values between 0.01 and 0.10.

In [10]:
rate = labels_df['anomaly_flag'].mean() * 100
print(f'Anomaly rate      : {rate:.2f}%')
print(f'features_df shape : {features_df.shape}')
print(f'labels_df shape   : {labels_df.shape}')
print()
print('Role D: pass features_df into Isolation Forest')
print('Role F: compare Isolation Forest output against labels_df to measure performance')

Anomaly rate      : 6.74%
features_df shape : (169539, 13)
labels_df shape   : (169539, 5)

Role D: pass features_df into Isolation Forest
Role F: compare Isolation Forest output against labels_df to measure performance


## 6. Optional – save outputs to CSV

Not saved automatically. Run this cell only when you want to persist the files.

In [11]:
# Uncomment to save:
save_outputs(features_df, labels_df, output_dir='../data/processed')

[label_builder] Saved features → ../data/processed/anomaly_features_v1.csv
[label_builder] Saved labels   → ../data/processed/anomaly_labels_v1.csv


## 7. Using the pipeline entry point (how AI008 calls this)

AI008's `run.py` calls `run()` directly. This is equivalent to all the steps above in one call.

In [12]:
features_df, labels_df = run(
    dataset_path=str(dataset_path),
    save=False
)
print(f'features_df: {features_df.shape}')
print(f'labels_df  : {labels_df.shape}')

[label_builder] Loading dataset: anomaly_detection_hourly_2020_2024.csv
[label_builder] Loaded 169,539 rows, 45 columns
[label_builder] Feature set prepared: 13 features, 169,539 records
[label_builder] Rule results:
[label_builder]   extreme_fire_intensity                     1,696 rows flagged
[label_builder]   high_confidence_extreme_fire               3,034 rows flagged
[label_builder]   cyber_spike_low_hazard                     1,593 rows flagged
[label_builder]   hazard_cyber_mismatch                      8,418 rows flagged
[label_builder]   active_online_threats                      2,939 rows flagged
[label_builder]   rare_region_active                            68 rows flagged
[label_builder]   cyber_volume_spike                         2,939 rows flagged

[label_builder] Label summary:
  Total rows : 169,539
  Anomalies  : 11,419  (6.74%)
  Normal     : 158,120
features_df: (169539, 13)
labels_df  : (169539, 5)
